# Avance 3 — Análisis Inferencial
## Variabilidad en el Tratamiento Hospitalario y sus Efectos en Mortalidad y Estadía:
## Neoplasias y Sepsis en el Sistema Público Chileno

**Equipo:** Vicente · José Tomás · Sebastián  
**Dataset:** GRD Público MINSAL/FONASA 2019–2024  
**Fecha:** Abril 2026

---

### Estructura del análisis

| Hipótesis | Variable dependiente | Método |
|-----------|---------------------|--------|
| **H1** | Intensidad de procedimientos (`n_procedimientos`) por hospital | Kruskal-Wallis + post-hoc Dunn-Bonferroni |
| **H2** | Mortalidad intrahospitalaria (`mortalidad_intrahospitalaria`) | Regresión Logística (OR + IC 95%) |
| **H3** | Días de estadía (`dias_estada`) | Regresión OLS con log-transformación |

Cada hipótesis se evalúa **por separado** para el grupo **Neoplasias** y el grupo **Sepsis**,
controlando por severidad clínica (peso GRD, nivel de severidad, edad).


---
## 0. Librerías y configuración


In [ ]:
import re
import warnings
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import (rankdata, norm as sp_norm,
                         chi2_contingency, fisher_exact)
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf
import statsmodels.api as sm
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120

# ── Parámetros globales ────────────────────────────────────────────────────
ALPHA         = 0.05       # Nivel de significancia
SEMILLA       = 42         # Semilla reproducible
MIN_CASOS_H   = 30         # Mínimo de casos por hospital para incluirlo
TOP_HOSP_KW   = 25         # Top N hospitales para Kruskal-Wallis / Dunn
TOP_HOSP_REG  = 15         # Top N hospitales para regresiones
P_OUTLIER     = 0.99       # Percentil de corte outliers días de estadía

# Rutas
ROOT_DIR  = Path('..')
CSV_CLEAN = ROOT_DIR / 'DATASET INICIAL' / 'df_clean_final_2019_2024.csv'
OUT_DIR   = Path('.') / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Librerías cargadas correctamente.')
print(f'Dataset:           {CSV_CLEAN.resolve()}')
print(f'Directorio salida: {OUT_DIR.resolve()}')


---
## 1. Carga y preparación del dataset

Se carga directamente el dataset limpio **`df_clean_final_2019_2024.csv`** generado
en el preprocesamiento (~441 k egresos GRD del sistema público chileno, 2019–2024).

Se filtran dos grupos diagnósticos según los códigos CIE-10 definidos en el proyecto:

| Grupo | Códigos CIE-10 | Diagnósticos |
|-------|---------------|--------------|
| **Neoplasia** | C50, C18–C20, C53, C34 | Mama, Colorrectal, Cuello uterino, Pulmón |
| **Sepsis** | A40, A41 | Estreptocócica, Otras sepsis |


In [ ]:
# ── Funciones auxiliares para tests categóricos ──────────────────────────

def fmt_p(p):
    """Formato APA 7 para p-valores."""
    if p < 1e-16: return '< 1e-16'
    if p < 0.001: return f'{p:.2e}'
    return f'{p:.4f}'

def cramers_v(table):
    """V de Cramér para tablas de contingencia (efecto chi-cuadrado)."""
    chi2 = chi2_contingency(table)[0]
    n    = np.asarray(table).sum()
    r, c = np.asarray(table).shape
    return np.sqrt(chi2 / (n * min(r - 1, c - 1)))

def interpret_cramers_v(v, r, c):
    """Interpreta V según dimensión de tabla (Cohen 1988 adaptado)."""
    k = min(r, c) - 1
    thresholds = {1: (0.10, 0.30), 2: (0.07, 0.21), 3: (0.06, 0.17)}
    lo, hi = thresholds.get(k, (0.05, 0.15))
    if v < lo:  return 'pequeño'
    if v < hi:  return 'moderado'
    return 'grande'

def standardized_residuals(table):
    """Residuos estandarizados para localizar celdas que impulsan el chi-cuadrado."""
    _, _, _, expected = chi2_contingency(table)
    obs = np.asarray(table, dtype=float)
    return pd.DataFrame((obs - expected) / np.sqrt(expected),
                        index=table.index, columns=table.columns)

print('Funciones auxiliares definidas: fmt_p, cramers_v, interpret_cramers_v, standardized_residuals.')


In [ ]:
# ── Carga del dataset limpio df_clean_final_2019_2024 ────────────────────
if not CSV_CLEAN.exists():
    raise FileNotFoundError(
        f'No se encontró: {CSV_CLEAN.resolve()}\n'
        'Verifica que el archivo esté en DATASET INICIAL/'
    )

df_base = pd.read_csv(CSV_CLEAN, low_memory=False)

# Convertir peso GRD (formato coma decimal → punto)
df_base['IR_29301_PESO'] = pd.to_numeric(
    df_base['IR_29301_PESO'].astype(str).str.replace(',', '.', regex=False),
    errors='coerce'
)

print(f'Dataset cargado: {len(df_base):,} registros')
print(f'Columnas disponibles: {list(df_base.columns)}')
print()
print('TIPOALTA — distribución:')
display(df_base['TIPOALTA'].value_counts().head(8).to_frame('n'))


In [ ]:
# ── Clasificar grupo_diagnostico: Neoplasia / Sepsis ─────────────────────
# Códigos CIE-10 definidos en el Avance 1 (sección 3.1 del informe)
PREFIJOS_NEOPLASIA = ('C50', 'C18', 'C19', 'C20', 'C53', 'C34')
PREFIJOS_SEPSIS    = ('A40', 'A41')

diag = df_base['DIAGNOSTICO1'].astype(str).str.upper().str.strip()

mask_neo = diag.str.startswith(PREFIJOS_NEOPLASIA)
mask_sep = diag.str.startswith(PREFIJOS_SEPSIS)

df_base['grupo_diagnostico'] = np.where(
    mask_neo, 'Neoplasia',
    np.where(mask_sep, 'Sepsis', 'Otro')
)

print('Distribución por grupo diagnóstico:')
display(df_base['grupo_diagnostico'].value_counts().to_frame('n'))

print('\nDiagnósticos más frecuentes (top 5 por grupo):')
for grp in ['Neoplasia', 'Sepsis']:
    top = (df_base[df_base['grupo_diagnostico'] == grp]['DIAGNOSTICO1']
           .value_counts().head(5))
    print(f'  {grp}: {dict(top)}')


In [ ]:
# ── Limpieza final y estandarización de columnas ─────────────────────────
df_clean = df_base[df_base['grupo_diagnostico'].isin(['Neoplasia','Sepsis'])].copy()

# Renombrar para compatibilidad con funciones estadísticas
df_clean = df_clean.rename(columns={
    'severidad':    'severidad_grd',
    'peso_grd':     'peso_relativo_grd',
    'DIAGNOSTICO1': 'diagnostico_principal',
})

# Mortalidad intrahospitalaria binaria (0/1) derivada de TIPOALTA
df_clean['mortalidad_intrahospitalaria'] = (
    df_clean['TIPOALTA'].astype(str).str.upper().str.contains('FALLECIDO', na=False)
).astype(int)

# Tipos numéricos
for col in ['dias_estada','edad','n_procedimientos','peso_relativo_grd','severidad_grd']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Eliminar nulos en variables críticas
vars_criticas = ['COD_HOSPITAL','dias_estada','n_procedimientos',
                 'mortalidad_intrahospitalaria','edad','severidad_grd','peso_relativo_grd']
df_clean = df_clean.dropna(subset=vars_criticas)

# Outliers: corte p99 en días de estadía
df_clean = df_clean[df_clean['dias_estada'] >= 0]
p99 = df_clean['dias_estada'].quantile(P_OUTLIER)
df_clean = df_clean[df_clean['dias_estada'] <= p99].copy()
df_clean['mortalidad_intrahospitalaria'] = df_clean['mortalidad_intrahospitalaria'].astype(int)

n_neo = (df_clean['grupo_diagnostico'] == 'Neoplasia').sum()
n_sep = (df_clean['grupo_diagnostico'] == 'Sepsis').sum()

print(f'Dataset final de inferencia: {len(df_clean):,} registros')
print(f'  Neoplasias : {n_neo:,}')
print(f'  Sepsis     : {n_sep:,}')
print(f'  Hospitales únicos      : {df_clean["COD_HOSPITAL"].nunique()}')
print(f'  Corte outlier p{int(P_OUTLIER*100)} días: {p99:.1f} días')
print(f'  Tasa mortalidad global : {df_clean["mortalidad_intrahospitalaria"].mean():.2%}')

# Subconjuntos por grupo diagnóstico
df_neo = df_clean[df_clean['grupo_diagnostico'] == 'Neoplasia'].copy()
df_sep = df_clean[df_clean['grupo_diagnostico'] == 'Sepsis'].copy()


---
## 2. Análisis de Variables Categóricas — Tests Chi-Cuadrado

Siguiendo la metodología del apunte de curso (Tests para variables binarias y categóricas),
se analizan cuatro escenarios antes de los modelos inferenciales principales:

| Escenario | Respuesta | Grupos | Tabla | Prueba |
|-----------|-----------|--------|-------|--------|
| **A** | `mortalidad_intrahospitalaria` (0/1) | Neoplasia vs Sepsis (2) | 2×2 | χ² + z proporciones |
| **B** | `mortalidad_intrahospitalaria` (0/1) | Tipo de ingreso (3) | 2×3 | χ² + post-hoc Bonferroni |
| **C** | `tipoalta_simple` (5 categorías) | Neoplasia vs Sepsis (2) | 2×5 | χ² + residuos estand. |
| **D** | `tipoalta_simple` (5 categorías) | Tipo de ingreso (3) | 3×5 | χ² + residuos estand. |

> **Criterio de Fisher**: se usa Fisher exacta en tabla 2×2 si alguna frecuencia esperada < 5.  
> **Tamaño del efecto**: V de Cramér. Umbrales para tabla 2×2: pequeño < 0.10, moderado 0.10–0.30, grande > 0.30.  
> **Reporte APA 7**: χ²(gl, N) = X.XX, p < .001, V = X.XX.


In [ ]:
# ── Preparar tipoalta_simple (5 categorías resumidas del egreso) ──────────
def simplificar_tipoalta(x):
    x = str(x).upper()
    if 'DOMICILIO' in x and 'HOSPITALIZ' not in x:
        return 'Domicilio'
    if 'FALLECIDO' in x:
        return 'Fallecido'
    if 'HOSPITALIZ' in x and 'DOMICILIO' in x:
        return 'Hosp. domiciliaria'
    if 'DERIVACI' in x:
        return 'Derivación'
    return 'Otras'

df_clean['tipoalta_simple'] = df_clean['TIPOALTA'].map(simplificar_tipoalta)
# Propagar a subconjuntos
df_neo['tipoalta_simple'] = df_clean.loc[df_neo.index, 'tipoalta_simple']
df_sep['tipoalta_simple'] = df_clean.loc[df_sep.index, 'tipoalta_simple']

print('Distribución de tipoalta_simple:')
display(df_clean['tipoalta_simple'].value_counts().to_frame('n'))

print('\nTasa de mortalidad intrahospitalaria por grupo diagnóstico:')
display(
    df_clean.groupby('grupo_diagnostico')['mortalidad_intrahospitalaria']
    .agg(n='count', fallecidos='sum', tasa='mean')
    .assign(tasa=lambda d: (d['tasa']*100).round(2))
    .rename(columns={'tasa':'tasa (%)'})
)


### 2.1 Escenario A — Mortalidad binaria × Grupo diagnóstico (2×2)

**Pregunta:** ¿La proporción de fallecimiento difiere significativamente entre Neoplasia y Sepsis?


In [ ]:
# ── Escenario A: mortalidad ~ grupo_diagnostico ───────────────────────────
tab_A = pd.crosstab(df_clean['grupo_diagnostico'],
                    df_clean['mortalidad_intrahospitalaria'])
tab_A.columns = ['No fallecido', 'Fallecido']

print('Tabla de contingencia (conteos):')
display(tab_A)
tab_A_prop = tab_A.div(tab_A.sum(axis=1), axis=0)
print('\nProporción de fallecidos (%):')
display((tab_A_prop * 100).round(2))

# ── Tests ──────────────────────────────────────────────────────────────────
chi2_A, p_A, dof_A, exp_A = chi2_contingency(tab_A)
min_exp_A = np.min(exp_A)

counts_A = tab_A['Fallecido'].values
nobs_A   = tab_A.sum(axis=1).values
z_A, pz_A = proportions_ztest(counts_A, nobs_A)

# Odds Ratio + IC95%
a, b = tab_A.iloc[0, 1], tab_A.iloc[0, 0]
c, d = tab_A.iloc[1, 1], tab_A.iloc[1, 0]
or_A  = (a / b) / (c / d)
se_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
or_lo = np.exp(np.log(or_A) - 1.96*se_or)
or_hi = np.exp(np.log(or_A) + 1.96*se_or)

# Fisher exacta (verificación)
_, p_fish_A = fisher_exact(tab_A.values)

v_A  = cramers_v(tab_A)
r_A, c_A_dim = tab_A.shape
ef_A = interpret_cramers_v(v_A, r_A, c_A_dim)

print(f'\n── Resultados Escenario A ──────────────────────────────────')
print(f'  Chi-cuadrado   χ²({dof_A}) = {chi2_A:.2f},  p = {fmt_p(p_A)}')
print(f'  Z proporciones  z = {z_A:.4f},  p = {fmt_p(pz_A)}')
print(f'  Fisher exacta   p = {fmt_p(p_fish_A)}  '
      f'(frec. mín. esperada = {min_exp_A:.1f})')
print(f'  V de Cramér = {v_A:.4f}  → efecto {ef_A}')
print(f'  Odds Ratio ({tab_A.index[0]} vs {tab_A.index[1]}) = {or_A:.4f}  '
      f'IC95% [{or_lo:.4f}, {or_hi:.4f}]')
print()
if p_A < ALPHA:
    print(f'  → SE RECHAZA H₀ (α = {ALPHA}): las proporciones de mortalidad difieren.')
else:
    print(f'  → No se rechaza H₀: diferencia no significativa.')

# Gráfico
fig, ax = plt.subplots(figsize=(7, 4))
(tab_A_prop['Fallecido']*100).plot(kind='bar', ax=ax,
                                    color=['steelblue','coral'], edgecolor='white')
ax.set_title(f'Mortalidad intrahospitalaria por grupo diagnóstico\n'
             f'χ²({dof_A}) = {chi2_A:.2f}, p {fmt_p(p_A)}, V = {v_A:.3f}')
ax.set_ylabel('Fallecidos (%)'); ax.set_xlabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for p_bar in ax.patches:
    ax.annotate(f'{p_bar.get_height():.2f}%',
                (p_bar.get_x()+p_bar.get_width()/2, p_bar.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / 'cat_A_mortalidad_grupo.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.2 Escenario B — Mortalidad binaria × Tipo de ingreso (2×3)

**Pregunta:** ¿La tasa de mortalidad intrahospitalaria varía según el tipo de ingreso
(URGENCIA / PROGRAMADA / OBSTETRICA)? Post-hoc Bonferroni si el global es significativo.


In [ ]:
# ── Escenario B: mortalidad ~ TIPO_INGRESO ────────────────────────────────
df_ti = df_clean[df_clean['TIPO_INGRESO'].isin(
    ['URGENCIA','PROGRAMADA','OBSTETRICA'])].copy()

tab_B = pd.crosstab(df_ti['TIPO_INGRESO'],
                    df_ti['mortalidad_intrahospitalaria'])
tab_B.columns = ['No fallecido', 'Fallecido']
tab_B = tab_B.reindex(['URGENCIA','PROGRAMADA','OBSTETRICA'])

print('Tabla de contingencia (conteos):')
display(tab_B)
tab_B_prop = tab_B.div(tab_B.sum(axis=1), axis=0)
print('\nProporción de fallecidos (%):')
display((tab_B_prop[['Fallecido']]*100).round(3))

# ── Chi-cuadrado global ───────────────────────────────────────────────────
chi2_B, p_B, dof_B, _ = chi2_contingency(tab_B)
v_B  = cramers_v(tab_B)
r_B, c_B_dim = tab_B.shape
ef_B = interpret_cramers_v(v_B, r_B, c_B_dim)

print(f'\n── Prueba global ─────────────────────────────────────────')
print(f'  Chi-cuadrado χ²({dof_B}) = {chi2_B:.2f},  p = {fmt_p(p_B)}')
print(f'  V de Cramér = {v_B:.4f}  → efecto {ef_B}')

# ── Post-hoc: z de dos proporciones + corrección Bonferroni ──────────────
rows_B = []
for g1, g2 in itertools.combinations(tab_B.index, 2):
    c1, n1 = tab_B.loc[g1,'Fallecido'], int(tab_B.loc[g1].sum())
    c2, n2 = tab_B.loc[g2,'Fallecido'], int(tab_B.loc[g2].sum())
    z, p   = proportions_ztest([c1, c2], [n1, n2])
    p1, p2 = c1/n1, c2/n2
    diff   = p1 - p2
    se     = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
    rows_B.append({'Grupo 1':g1, 'Grupo 2':g2,
                   'Prop 1 (%)': round(p1*100,3), 'Prop 2 (%)': round(p2*100,3),
                   'Diferencia (pp)': round(diff*100,3),
                   'IC95 inf (pp)':   round((diff-1.96*se)*100,3),
                   'IC95 sup (pp)':   round((diff+1.96*se)*100,3),
                   'z': round(z,4), 'p_raw': p})

df_ph_B = pd.DataFrame(rows_B)
_, p_adj_B, _, _ = multipletests(df_ph_B['p_raw'], method='bonferroni')
df_ph_B['p_adj (Bonf)'] = p_adj_B.round(6)
df_ph_B['sig'] = df_ph_B['p_adj (Bonf)'].apply(
    lambda p: '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns')))

print('\nPost-hoc pareado (z de dos proporciones + Bonferroni):')
display(df_ph_B.drop(columns=['p_raw']))

# Gráfico
fig, ax = plt.subplots(figsize=(8, 4))
colores_ti = ['salmon','steelblue','lightgreen']
(tab_B_prop['Fallecido']*100).plot(kind='bar', ax=ax,
                                    color=colores_ti, edgecolor='white')
ax.set_title(f'Mortalidad intrahospitalaria por tipo de ingreso\n'
             f'χ²({dof_B}) = {chi2_B:.2f}, p {fmt_p(p_B)}, V = {v_B:.3f}')
ax.set_ylabel('Fallecidos (%)'); ax.set_xlabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for p_bar in ax.patches:
    ax.annotate(f'{p_bar.get_height():.2f}%',
                (p_bar.get_x()+p_bar.get_width()/2, p_bar.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(OUT_DIR / 'cat_B_mortalidad_tipoing.png', dpi=150, bbox_inches='tight')
plt.show()
df_ph_B.to_csv(OUT_DIR / 'cat_B_posthoc_bonferroni.csv', index=False)


### 2.3 Escenario C — Tipo de alta (categórica) × Grupo diagnóstico (2×5)

**Pregunta:** ¿La distribución del tipo de egreso difiere entre pacientes oncológicos y sépticos?
Los residuos estandarizados localizan exactamente qué categorías impulsan el χ².


In [ ]:
# ── Escenario C: tipoalta_simple ~ grupo_diagnostico ─────────────────────
orden_cat = ['Domicilio','Fallecido','Hosp. domiciliaria','Derivación','Otras']

tab_C = pd.crosstab(df_clean['grupo_diagnostico'], df_clean['tipoalta_simple'])
tab_C = tab_C[[c for c in orden_cat if c in tab_C.columns]]

print('Tabla de contingencia (conteos):')
display(tab_C)
print('\nProporciones por fila (%):')
display((tab_C.div(tab_C.sum(axis=1), axis=0)*100).round(2))

chi2_C, p_C, dof_C, _ = chi2_contingency(tab_C)
v_C  = cramers_v(tab_C)
r_C, c_C_dim = tab_C.shape
ef_C = interpret_cramers_v(v_C, r_C, c_C_dim)
resid_C = standardized_residuals(tab_C)

print(f'\n── Resultados Escenario C ──────────────────────────────────')
print(f'  Chi-cuadrado χ²({dof_C}) = {chi2_C:.2f},  p = {fmt_p(p_C)}')
print(f'  V de Cramér = {v_C:.4f}  → efecto {ef_C}')
print()
print('Residuos estandarizados (|r| > 2 → contribución relevante al χ²):')
display(resid_C.round(2))

# Heatmap residuos
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(resid_C, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-10, vmax=10, linewidths=0.5, ax=ax)
ax.set_title(f'Residuos estandarizados: tipo de alta × grupo diagnóstico\n'
             f'χ²({dof_C}) = {chi2_C:.2f}, p {fmt_p(p_C)}, V = {v_C:.3f}')
plt.tight_layout()
plt.savefig(OUT_DIR / 'cat_C_residuos_tipoalta_grupo.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.4 Escenario D — Tipo de alta (categórica) × Tipo de ingreso (3×5)

**Pregunta:** ¿La distribución de egresos varía según la vía de entrada al hospital?


In [ ]:
# ── Escenario D: tipoalta_simple ~ TIPO_INGRESO ──────────────────────────
df_ti2  = df_clean[df_clean['TIPO_INGRESO'].isin(
    ['URGENCIA','PROGRAMADA','OBSTETRICA'])].copy()
tab_D   = pd.crosstab(df_ti2['TIPO_INGRESO'], df_ti2['tipoalta_simple'])
tab_D   = tab_D[[c for c in orden_cat if c in tab_D.columns]]
tab_D   = tab_D.reindex([r for r in ['URGENCIA','PROGRAMADA','OBSTETRICA']
                          if r in tab_D.index])

print('Proporciones por fila (%):')
display((tab_D.div(tab_D.sum(axis=1), axis=0)*100).round(2))

chi2_D, p_D, dof_D, _ = chi2_contingency(tab_D)
v_D   = cramers_v(tab_D)
r_D, c_D_dim = tab_D.shape
ef_D  = interpret_cramers_v(v_D, r_D, c_D_dim)
resid_D = standardized_residuals(tab_D)

print(f'\n── Resultados Escenario D ──────────────────────────────────')
print(f'  Chi-cuadrado χ²({dof_D}) = {chi2_D:.2f},  p = {fmt_p(p_D)}')
print(f'  V de Cramér = {v_D:.4f}  → efecto {ef_D}')
print()
print('Residuos estandarizados:')
display(resid_D.round(2))

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(resid_D, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-15, vmax=15, linewidths=0.5, ax=ax)
ax.set_title(f'Residuos estandarizados: tipo de alta × tipo de ingreso\n'
             f'χ²({dof_D}) = {chi2_D:.2f}, p {fmt_p(p_D)}, V = {v_D:.3f}')
plt.tight_layout()
plt.savefig(OUT_DIR / 'cat_D_residuos_tipoalta_tipoing.png', dpi=150, bbox_inches='tight')
plt.show()


### Interpretación — Análisis Categórico

**Cómo leer los residuos estandarizados:**
- |r| < 2 → celda dentro de lo esperado bajo H₀
- 2 ≤ |r| < 4 → contribución notable; r > 0 = más observados de lo esperado
- |r| ≥ 4 → contribución muy fuerte (señal robusta de asociación)

**Sobre el V de Cramér:** con N ≈ 40 k, incluso efectos pequeños generan p ≪ 0.001.
Siempre reportar junto al p-valor: *χ²(gl, N) = X.XX, p < .001, V = X.XX*.

**Sobre el OR (Escenario A):** OR < 1 indica que el primer grupo tiene menor odds de fallecer
respecto al segundo. El IC95% permite evaluar precisión y significancia práctica.


---
## 3. Hipótesis 1 — Intensidad de procedimientos por hospital

**H₀:** La distribución del número de procedimientos por hospitalización (`n_procedimientos`)
no difiere significativamente entre hospitales, para cada grupo diagnóstico.

**H₁:** Existe variabilidad estadísticamente significativa en la intensidad de procedimientos
entre hospitales, lo que evidencia asimetría en la gestión clínica del sistema público chileno.

**Estrategia:**
1. Shapiro-Wilk para confirmar no-normalidad (justifica test no paramétrico).
2. Kruskal-Wallis → estadístico H, p-valor, tamaño del efecto ε².
3. Si p < 0.05: Dunn post-hoc con corrección de Bonferroni → identificar pares de hospitales que difieren.


In [ ]:
# ── Función: test de Shapiro-Wilk con submuestra fija ────────────────────
def test_shapiro(arr, semilla=SEMILLA, n_max=5000):
    """Aplica Shapiro-Wilk; si n > n_max, extrae submuestra con semilla fija."""
    arr = np.asarray(arr)
    arr = arr[~np.isnan(arr)]
    rng = np.random.default_rng(semilla)
    if len(arr) > n_max:
        muestra = rng.choice(arr, size=n_max, replace=False)
        nota = f'(submuestra aleatoria n={n_max:,}, semilla={semilla})'
    else:
        muestra = arr
        nota = f'(muestra completa n={len(arr):,})'
    W, p = stats.shapiro(muestra)
    return W, p, nota


# ── Función: test de Kruskal-Wallis con ε² ────────────────────────────────
def kruskal_wallis(grupos_vals, nombres_grupos):
    """
    Recibe lista de arrays y lista de nombres.
    Devuelve dict con H, p, epsilon_cuadrado, k, N.
    """
    k = len(grupos_vals)
    N = sum(len(g) for g in grupos_vals)
    H, p = stats.kruskal(*grupos_vals)
    # Epsilon-cuadrado (Tomczak & Tomczak, 2014)
    eps2 = (H - k + 1) / (N - k)
    return {'H': H, 'p': p, 'epsilon2': eps2, 'k': k, 'N': N,
            'grupos': nombres_grupos}


# ── Función: test de Dunn con corrección de Bonferroni ────────────────────
def dunn_bonferroni(grupos_vals, nombres_grupos):
    """
    Implementación propia del test post-hoc de Dunn.
    Devuelve DataFrame con columnas: grupo_i, grupo_j, z, p_raw, p_adj, significativo.
    Referencia: Dunn (1964), corrección Bonferroni.
    """
    k  = len(grupos_vals)
    ns = [len(g) for g in grupos_vals]
    N  = sum(ns)

    # Rango global de todos los datos
    all_data  = np.concatenate(grupos_vals)
    all_ranks = rankdata(all_data)

    # Rango medio por grupo
    idx = 0
    mean_ranks = []
    for n in ns:
        mean_ranks.append(np.mean(all_ranks[idx:idx+n]))
        idx += n

    # Corrección por empates
    _, counts = np.unique(all_data, return_counts=True)
    tie_corr  = np.sum(counts**3 - counts)
    sigma2    = (N*(N+1)/12 - tie_corr/(12*(N-1)))

    comparaciones = list(itertools.combinations(range(k), 2))
    resultados = []
    for i, j in comparaciones:
        se = np.sqrt(sigma2 * (1/ns[i] + 1/ns[j]))
        z  = abs(mean_ranks[i] - mean_ranks[j]) / se
        p  = 2 * (1 - sp_norm.cdf(z))
        resultados.append({
            'grupo_i': nombres_grupos[i],
            'grupo_j': nombres_grupos[j],
            'z': z,
            'p_raw': p,
        })

    m = len(resultados)  # n° de comparaciones para Bonferroni
    for r in resultados:
        r['p_adj']       = min(r['p_raw'] * m, 1.0)
        r['significativo'] = r['p_adj'] < ALPHA

    return pd.DataFrame(resultados)

print('Funciones estadísticas definidas: shapiro, kruskal_wallis, dunn_bonferroni.')


In [ ]:
# ── Preparar grupos: top hospitales por volumen ───────────────────────────
def preparar_grupos_kw(df, variable='n_procedimientos', top_n=TOP_HOSP_KW, min_n=MIN_CASOS_H):
    """
    Filtra los top_n hospitales con mayor volumen de casos (mín. min_n).
    Devuelve (grupos_vals, nombres_grupos, df_filtrado).
    """
    conteo = df['COD_HOSPITAL'].value_counts()
    hosp_ok = conteo[conteo >= min_n].head(top_n).index
    df_f = df[df['COD_HOSPITAL'].isin(hosp_ok)].copy()
    grupos_vals  = [df_f.loc[df_f['COD_HOSPITAL']==h, variable].dropna().values
                    for h in hosp_ok]
    grupos_vals  = [g for g in grupos_vals if len(g) > 0]
    nombres      = [str(h) for h in hosp_ok[:len(grupos_vals)]]
    return grupos_vals, nombres, df_f

print('Función preparar_grupos_kw definida.')


### 3.1 Neoplasias — verificación de normalidad


In [ ]:
# ── Shapiro-Wilk: Neoplasias ──────────────────────────────────────────────
W_neo, p_sw_neo, nota_neo = test_shapiro(df_neo['n_procedimientos'].dropna().values)

print('VERIFICACIÓN DE NORMALIDAD — Grupo Neoplasias')
print(f'Variable: n_procedimientos   {nota_neo}')
print(f'  Estadístico W = {W_neo:.6f}')
print(f'  p-valor       = {p_sw_neo:.4e}')
print()
if p_sw_neo < ALPHA:
    print(f'  → Se RECHAZA la hipótesis de normalidad (α = {ALPHA}).')
    print('    Justificación para usar Kruskal-Wallis: la distribución es asimétrica.')
else:
    print(f'  → No se rechaza la hipótesis de normalidad (α = {ALPHA}).')
    print('    Se evalúa Kruskal-Wallis por tamaño de muestra y heterogeneidad de grupos.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_neo['n_procedimientos'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de n_procedimientos\n(Neoplasias)')
axes[0].set_xlabel('Número de procedimientos')
axes[0].set_ylabel('Frecuencia')
stats.probplot(df_neo['n_procedimientos'].dropna().values, plot=axes[1])
axes[1].set_title('Q-Q Plot — n_procedimientos (Neoplasias)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'h1_neo_normalidad.png', dpi=150, bbox_inches='tight')
plt.show()


### 3.2 Neoplasias — Kruskal-Wallis


In [ ]:
# ── Kruskal-Wallis: n_procedimientos ~ COD_HOSPITAL (Neoplasias) ──────────
grupos_neo, nombres_neo, df_neo_kw = preparar_grupos_kw(df_neo)

res_kw_neo = kruskal_wallis(grupos_neo, nombres_neo)

print('PRUEBA DE KRUSKAL-WALLIS — Grupo Neoplasias')
print(f'  Variable dependiente: n_procedimientos')
print(f'  Factor de agrupación: COD_HOSPITAL')
print(f'  Hospitales analizados (top {TOP_HOSP_KW}, min {MIN_CASOS_H} casos): {res_kw_neo["k"]}')
print(f'  N total:              {res_kw_neo["N"]:,}')
print(f'  Estadístico H:        {res_kw_neo["H"]:.4f}')
print(f'  p-valor:              {res_kw_neo["p"]:.4e}')
print(f'  Epsilon-cuadrado (ε²): {res_kw_neo["epsilon2"]:.4f}')
print()
if res_kw_neo['p'] < ALPHA:
    eps = res_kw_neo['epsilon2']
    mag = 'grande' if eps >= 0.14 else ('moderado' if eps >= 0.06 else 'pequeño')
    print(f'  → SE RECHAZA H₀ (α = {ALPHA}). Existe variabilidad significativa entre hospitales.')
    print(f'    Tamaño del efecto ε² = {eps:.4f} → efecto {mag}.')
    print(f'    Se procede al análisis post-hoc de Dunn con corrección de Bonferroni.')
else:
    print(f'  → No se rechaza H₀ (p = {res_kw_neo["p"]:.4f}). Sin diferencias significativas.')


### 3.3 Neoplasias — Post-hoc Dunn-Bonferroni


In [ ]:
# ── Dunn post-hoc: Neoplasias ─────────────────────────────────────────────
if res_kw_neo['p'] < ALPHA:
    dunn_neo = dunn_bonferroni(grupos_neo, nombres_neo)

    n_sig_neo = dunn_neo['significativo'].sum()
    print(f'POST-HOC DUNN-BONFERRONI — Neoplasias')
    print(f'  Total comparaciones pareadas: {len(dunn_neo)}')
    print(f'  Pares con diferencia significativa (p_adj < {ALPHA}): {n_sig_neo}')
    print()
    print('Pares significativos:')
    sig_pares = dunn_neo[dunn_neo['significativo']].sort_values('p_adj')
    display(sig_pares[['grupo_i','grupo_j','z','p_raw','p_adj']].round(4).head(20))

    # Heatmap de p-valores ajustados (top 15 hospitales)
    top15_neo = nombres_neo[:15]
    p_matrix_neo = pd.DataFrame(np.ones((len(top15_neo), len(top15_neo))),
                                index=top15_neo, columns=top15_neo)
    for _, row in dunn_neo.iterrows():
        if row['grupo_i'] in top15_neo and row['grupo_j'] in top15_neo:
            p_matrix_neo.loc[row['grupo_i'], row['grupo_j']] = row['p_adj']
            p_matrix_neo.loc[row['grupo_j'], row['grupo_i']] = row['p_adj']

    fig, ax = plt.subplots(figsize=(12, 10))
    mask_diag = np.eye(len(top15_neo), dtype=bool)
    sns.heatmap(
        p_matrix_neo.astype(float), mask=mask_diag,
        cmap='RdYlGn_r', vmin=0, vmax=0.1, annot=True, fmt='.3f',
        linewidths=0.5, ax=ax
    )
    ax.set_title(
        'p-valores ajustados (Dunn-Bonferroni) — Neoplasias\n'
        'Verde = no significativo · Rojo = diferencia significativa (p_adj < 0.05)',
        fontsize=11
    )
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'h1_neo_dunn_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    dunn_neo.to_csv(OUT_DIR / 'h1_neo_dunn_resultados.csv', index=False)
    print('Heatmap y tabla CSV guardados en advance_3/outputs/')
else:
    print('Kruskal-Wallis no significativo: no se ejecuta Dunn post-hoc.')
    dunn_neo = pd.DataFrame()


### 3.4 Sepsis — Shapiro-Wilk, Kruskal-Wallis y Dunn-Bonferroni


In [ ]:
# ── Shapiro-Wilk: Sepsis ──────────────────────────────────────────────────
W_sep, p_sw_sep, nota_sep = test_shapiro(df_sep['n_procedimientos'].dropna().values)

print('VERIFICACIÓN DE NORMALIDAD — Grupo Sepsis')
print(f'Variable: n_procedimientos   {nota_sep}')
print(f'  Estadístico W = {W_sep:.6f}')
print(f'  p-valor       = {p_sw_sep:.4e}')
if p_sw_sep < ALPHA:
    print(f'  → Se RECHAZA la hipótesis de normalidad. Uso de Kruskal-Wallis justificado.')

# ── Kruskal-Wallis: Sepsis ────────────────────────────────────────────────
grupos_sep, nombres_sep, df_sep_kw = preparar_grupos_kw(df_sep)

res_kw_sep = kruskal_wallis(grupos_sep, nombres_sep)

print()
print('PRUEBA DE KRUSKAL-WALLIS — Grupo Sepsis')
print(f'  Hospitales analizados: {res_kw_sep["k"]}   N total: {res_kw_sep["N"]:,}')
print(f'  Estadístico H: {res_kw_sep["H"]:.4f}   p-valor: {res_kw_sep["p"]:.4e}')
print(f'  Epsilon-cuadrado (ε²): {res_kw_sep["epsilon2"]:.4f}')
if res_kw_sep['p'] < ALPHA:
    eps = res_kw_sep['epsilon2']
    mag = 'grande' if eps >= 0.14 else ('moderado' if eps >= 0.06 else 'pequeño')
    print(f'  → SE RECHAZA H₀. Efecto {mag} (ε² = {eps:.4f}).')

# ── Dunn post-hoc: Sepsis ─────────────────────────────────────────────────
if res_kw_sep['p'] < ALPHA:
    dunn_sep = dunn_bonferroni(grupos_sep, nombres_sep)
    n_sig_sep = dunn_sep['significativo'].sum()
    print(f'\nPOST-HOC DUNN-BONFERRONI — Sepsis')
    print(f'  Comparaciones totales: {len(dunn_sep)}   Significativas: {n_sig_sep}')
    display(dunn_sep[dunn_sep['significativo']].sort_values('p_adj')[['grupo_i','grupo_j','z','p_raw','p_adj']].round(4).head(20))
    dunn_sep.to_csv(OUT_DIR / 'h1_sep_dunn_resultados.csv', index=False)
else:
    dunn_sep = pd.DataFrame()
    print('Kruskal-Wallis no significativo: no se ejecuta Dunn post-hoc para Sepsis.')


### Interpretación — Hipótesis 1

La prueba de Kruskal-Wallis evalúa si la distribución de `n_procedimientos` es idéntica
entre hospitales (H₀) o si al menos un hospital difiere significativamente (H₁).
El tamaño del efecto **ε²** indica la proporción de varianza explicada por el factor hospital:
ε² < 0.06 → efecto pequeño · 0.06–0.14 → moderado · > 0.14 → grande.

El análisis post-hoc de **Dunn con corrección de Bonferroni** identifica los pares de hospitales
con diferencias significativas, controlando la tasa de error familiar (FWER ≤ α).

> **TODO — José Tomás:**  
> Interpreta el tamaño del efecto ε² en el contexto clínico. ¿Es esperable que hospitales
> de mayor complejidad realicen más procedimientos aunque el GRD sea comparable?
> ¿Podría la variabilidad reflejar diferencias en disponibilidad de pabellones o insumos?

> **TODO — Sebastián:**  
> Identifica los 3-5 pares de hospitales con mayor diferencia en el heatmap de Dunn.
> ¿Hay un patrón geográfico (norte/sur) o de nivel de complejidad (alta vs. baja)?  
> *Sección "Resultados — H1" del informe Word. Citar: Kruskal (1952), Dunn (1964).*


---
## 4. Hipótesis 2 — Mortalidad intrahospitalaria (Regresión Logística)

**H₀:** El número de procedimientos no tiene efecto sobre la probabilidad de mortalidad
intrahospitalaria, controlando por severidad, edad y hospital.

**H₁:** Una mayor intensidad de procedimientos se asocia con una reducción
(o aumento, según grupo) de la probabilidad de muerte intrahospitalaria.

**Modelo:**
```
logit(mortalidad_intrahospitalaria) =
    β₀ + β₁·n_procedimientos + β₂·edad + β₃·severidad_grd
    + β₄·peso_relativo_grd + Σ γₖ·C(COD_HOSPITAL)ₖ + ε
```

Se ajusta por separado para **Neoplasias** y **Sepsis**.
Se reportan **Odds Ratios (OR)** con **Intervalos de Confianza al 95%** y **Pseudo-R² de McFadden**.


In [ ]:
# ── Función: ajustar modelo logístico y extraer OR + IC95% ────────────────
def ajustar_logit(df_grp, nombre_grupo, top_n=TOP_HOSP_REG, min_n=MIN_CASOS_H):
    """
    Ajusta un modelo logístico con efectos fijos por hospital.
    Devuelve (resultado_statsmodels, tabla_OR_DataFrame).
    """
    # Filtrar hospitales con suficiente volumen
    conteo = df_grp['COD_HOSPITAL'].value_counts()
    hosp_ok = conteo[conteo >= min_n].head(top_n).index
    df_m = df_grp[df_grp['COD_HOSPITAL'].isin(hosp_ok)].copy()
    df_m['COD_HOSPITAL'] = df_m['COD_HOSPITAL'].astype('category')

    # Verificar variabilidad en la variable dependiente
    tasa_mort = df_m['mortalidad_intrahospitalaria'].mean()
    if tasa_mort == 0 or tasa_mort == 1:
        print(f'  [{nombre_grupo}] Mortalidad constante ({tasa_mort:.2%}). Modelo no estimable.')
        return None, pd.DataFrame()

    formula = (
        'mortalidad_intrahospitalaria ~ '
        'n_procedimientos + edad + severidad_grd + peso_relativo_grd + '
        'C(COD_HOSPITAL)'
    )

    try:
        modelo = smf.logit(formula, data=df_m).fit(
            method='bfgs', maxiter=500, disp=False
        )
    except Exception as e:
        print(f'  [{nombre_grupo}] Error en ajuste: {e}')
        return None, pd.DataFrame()

    # Extraer OR e IC95%
    coefs    = modelo.params
    ic       = modelo.conf_int(alpha=0.05)
    pvals    = modelo.pvalues

    tabla = pd.DataFrame({
        'OR':     np.exp(coefs),
        'IC_inf': np.exp(ic[0]),
        'IC_sup': np.exp(ic[1]),
        'p_valor': pvals,
    })
    tabla['sig'] = tabla['p_valor'].apply(
        lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    )
    tabla['grupo']     = nombre_grupo
    tabla['pseudo_R2'] = modelo.prsquared  # McFadden

    print(f'\n  Grupo: {nombre_grupo}')
    print(f'  N ajuste: {int(modelo.nobs):,}   |   Tasa mortalidad: {tasa_mort:.2%}')
    print(f'  Pseudo-R² (McFadden): {modelo.prsquared:.4f}')
    print(f'  Log-Likelihood: {modelo.llf:.2f}   |   AIC: {modelo.aic:.2f}')

    return modelo, tabla

print('Función ajustar_logit definida.')


### 4.1 Neoplasias — Regresión Logística


In [ ]:
print('=' * 65)
print('REGRESIÓN LOGÍSTICA — GRUPO NEOPLASIAS')
print('=' * 65)
modelo_logit_neo, tabla_or_neo = ajustar_logit(df_neo, 'Neoplasia')

if modelo_logit_neo is not None:
    # Mostrar solo coeficientes de interés (excluir efectos fijos de hospital)
    vars_interes = ['n_procedimientos', 'edad', 'severidad_grd', 'peso_relativo_grd']
    filas_interes = [idx for idx in tabla_or_neo.index
                     if any(v in str(idx) for v in vars_interes)]
    display(Markdown('**Odds Ratios — Variables principales (Neoplasias):**'))
    display(tabla_or_neo.loc[filas_interes, ['OR','IC_inf','IC_sup','p_valor','sig']].round(4))
    tabla_or_neo.to_csv(OUT_DIR / 'h2_neo_logit_OR.csv')
    print('\nTabla completa guardada: advance_3/outputs/h2_neo_logit_OR.csv')


#### Interpretación de los Odds Ratios — Neoplasias

El Odds Ratio de `n_procedimientos` cuantifica el cambio en la probabilidad de morir
asociado a cada procedimiento adicional, manteniendo constantes la edad, la severidad
clínica (GRD) y el hospital de atención.

- **OR < 1**: cada procedimiento adicional se asocia a una *reducción* del riesgo de muerte
  (posible efecto terapéutico).
- **OR > 1**: cada procedimiento adicional se asocia a un *aumento* del riesgo
  (posible marcador de deterioro clínico o intervenciones de rescate).
- El IC 95% indica la precisión del estimado; si no cruza 1.0, el efecto es estadísticamente
  significativo al nivel α = 0.05.

> **TODO — Vicente:**  
> Interpreta el OR de `n_procedimientos` en Neoplasias desde la perspectiva oncológica.
> ¿Es esperable que más procedimientos reduzcan la mortalidad en cáncer (terapia activa)
> o aumenten el riesgo (caso terminal con intervenciones de soporte)?
> Compara con la literatura sobre intensidad de tratamiento en oncología.
> *Sección "Resultados — H2" del Word. Formato APA 7: OR = X.XX, IC95% [X.XX, X.XX], p < .001.*


### 4.2 Sepsis — Regresión Logística


In [ ]:
print('=' * 65)
print('REGRESIÓN LOGÍSTICA — GRUPO SEPSIS')
print('=' * 65)
modelo_logit_sep, tabla_or_sep = ajustar_logit(df_sep, 'Sepsis')

if modelo_logit_sep is not None:
    filas_interes_sep = [idx for idx in tabla_or_sep.index
                         if any(v in str(idx)
                                for v in ['n_procedimientos','edad','severidad_grd','peso_relativo_grd'])]
    display(Markdown('**Odds Ratios — Variables principales (Sepsis):**'))
    display(tabla_or_sep.loc[filas_interes_sep, ['OR','IC_inf','IC_sup','p_valor','sig']].round(4))
    tabla_or_sep.to_csv(OUT_DIR / 'h2_sep_logit_OR.csv')
    print('Tabla guardada: advance_3/outputs/h2_sep_logit_OR.csv')


In [ ]:
# ── Forest plot comparativo: OR de n_procedimientos ──────────────────────
grupos_logit = []
for nombre, tabla, modelo in [
    ('Neoplasia', tabla_or_neo, modelo_logit_neo),
    ('Sepsis',    tabla_or_sep, modelo_logit_sep),
]:
    if modelo is not None and 'n_procedimientos' in tabla.index:
        r = tabla.loc['n_procedimientos']
        grupos_logit.append({
            'grupo': nombre,
            'OR': r['OR'], 'IC_inf': r['IC_inf'], 'IC_sup': r['IC_sup'],
            'p': r['p_valor']
        })

if grupos_logit:
    df_forest = pd.DataFrame(grupos_logit)
    fig, ax = plt.subplots(figsize=(8, 4))
    colores = ['steelblue','coral']
    for i, row in df_forest.iterrows():
        err_low  = row['OR'] - row['IC_inf']
        err_high = row['IC_sup'] - row['OR']
        ax.errorbar(
            x=row['OR'], y=i,
            xerr=[[err_low],[err_high]],
            fmt='o', color=colores[i], markersize=10, capsize=5, linewidth=2
        )
        ax.text(row['IC_sup'] + 0.01, i,
                f"OR={row['OR']:.3f} p={row['p']:.3e}", va='center', fontsize=9)
    ax.axvline(x=1, color='black', linestyle='--', linewidth=1)
    ax.set_yticks(range(len(df_forest)))
    ax.set_yticklabels(df_forest['grupo'])
    ax.set_xlabel('Odds Ratio de n_procedimientos')
    ax.set_title('Forest Plot — OR de n_procedimientos sobre Mortalidad\n(IC 95%, ajustado por edad, severidad GRD y hospital)')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'h2_forest_plot_OR.png', dpi=150, bbox_inches='tight')
    plt.show()


> **TODO — José Tomás:**  
> Compara los OR de Neoplasias y Sepsis en el forest plot.
> ¿En cuál condición los procedimientos tienen mayor impacto sobre la mortalidad?
> ¿El Pseudo-R² de McFadden es adecuado para ambos modelos (referencia: > 0.2 = buen ajuste)?
> Documenta las limitaciones del modelo (colinealidad hospital-severidad, eventos raros).
> *APA 7: χ²({gl}) = X.XX, p < .001, Pseudo-R² = X.XX.*


---
## 5. Hipótesis 3 — Días de estadía (Regresión OLS)

**H₀:** El número de procedimientos no predice significativamente la duración
de la hospitalización, una vez controlado por la severidad clínica y el hospital.

**H₁:** Una mayor intensidad de procedimientos se asocia de forma independiente
con una mayor duración de la estadía hospitalaria, incluso controlando por GRD.

**Modelo (con log-transformación si asimetría es pronunciada):**
```
log(1 + dias_estada) =
    β₀ + β₁·n_procedimientos + β₂·edad + β₃·severidad_grd
    + β₄·peso_relativo_grd + Σ γₖ·C(COD_HOSPITAL)ₖ + ε
```

> **Nota metodológica:** Se aplica `log(1 + dias_estada)` si la asimetría es > 1.0,
> transformando la VD para normalizar los residuos. Los coeficientes β representan
> entonces **semi-elasticidades**: un aumento de 1 unidad en el predictor está
> asociado a un cambio de β × 100% en los días de estadía.


In [ ]:
# ── Verificar asimetría y decidir transformación ──────────────────────────
asim_neo = df_neo['dias_estada'].skew()
asim_sep = df_sep['dias_estada'].skew()
UMBRAL_ASIMETRIA = 1.0

print('DIAGNÓSTICO DE ASIMETRÍA — dias_estada')
print(f'  Neoplasias: asimetría = {asim_neo:.3f}   '
      f'→ {"log-transform" if abs(asim_neo) > UMBRAL_ASIMETRIA else "sin transformación"}')
print(f'  Sepsis:     asimetría = {asim_sep:.3f}   '
      f'→ {"log-transform" if abs(asim_sep) > UMBRAL_ASIMETRIA else "sin transformación"}')

# Crear variable transformada
df_neo['log_dias'] = np.log1p(df_neo['dias_estada'])
df_sep['log_dias'] = np.log1p(df_sep['dias_estada'])

VD_NEO = 'log_dias' if abs(asim_neo) > UMBRAL_ASIMETRIA else 'dias_estada'
VD_SEP = 'log_dias' if abs(asim_sep) > UMBRAL_ASIMETRIA else 'dias_estada'

# Histogramas: original vs transformada
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
for ax, data, titulo in [
    (axes[0,0], df_neo['dias_estada'],  'Neoplasias — dias_estada (original)'),
    (axes[0,1], df_neo['log_dias'],     'Neoplasias — log(1+dias) [transformada]'),
    (axes[1,0], df_sep['dias_estada'],  'Sepsis — dias_estada (original)'),
    (axes[1,1], df_sep['log_dias'],     'Sepsis — log(1+dias) [transformada]'),
]:
    ax.hist(data.dropna(), bins=40, color='slateblue', edgecolor='white', alpha=0.8)
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frecuencia')
plt.suptitle('Distribución de días de estadía: original vs. transformada', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'h3_asimetria_dias.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Función: ajustar OLS y extraer coeficientes + IC95% ──────────────────
def ajustar_ols(df_grp, var_dep, nombre_grupo, top_n=TOP_HOSP_REG, min_n=MIN_CASOS_H):
    """
    Ajusta OLS con efectos fijos por hospital.
    Devuelve (resultado_statsmodels, tabla_coef_DataFrame).
    """
    conteo = df_grp['COD_HOSPITAL'].value_counts()
    hosp_ok = conteo[conteo >= min_n].head(top_n).index
    df_m = df_grp[df_grp['COD_HOSPITAL'].isin(hosp_ok)].copy()
    df_m['COD_HOSPITAL'] = df_m['COD_HOSPITAL'].astype('category')

    formula = (
        f'{var_dep} ~ '
        'n_procedimientos + edad + severidad_grd + peso_relativo_grd + '
        'C(COD_HOSPITAL)'
    )

    try:
        modelo = smf.ols(formula, data=df_m).fit(cov_type='HC3')  # Errores robustos
    except Exception as e:
        print(f'  [{nombre_grupo}] Error en OLS: {e}')
        return None, pd.DataFrame()

    ic   = modelo.conf_int(alpha=0.05)
    pvals = modelo.pvalues

    tabla = pd.DataFrame({
        'coef':   modelo.params,
        'IC_inf': ic[0],
        'IC_sup': ic[1],
        'p_valor': pvals,
    })
    tabla['sig'] = tabla['p_valor'].apply(
        lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    )
    tabla['grupo'] = nombre_grupo
    tabla['R2_ajustado'] = modelo.rsquared_adj
    tabla['var_dep']     = var_dep

    print(f'\n  Grupo: {nombre_grupo}   VD: {var_dep}')
    print(f'  N ajuste: {int(modelo.nobs):,}   R² ajustado: {modelo.rsquared_adj:.4f}')
    print(f'  F({int(modelo.df_model)}, {int(modelo.df_resid)}) = {modelo.fvalue:.2f}   '
          f'p-valor (F): {modelo.f_pvalue:.4e}')

    return modelo, tabla

print('Función ajustar_ols definida.')


### 5.1 Neoplasias — Regresión OLS


In [ ]:
print('=' * 65)
print('REGRESIÓN OLS — GRUPO NEOPLASIAS')
print('=' * 65)
modelo_ols_neo, tabla_coef_neo = ajustar_ols(df_neo, VD_NEO, 'Neoplasia')

if modelo_ols_neo is not None:
    vars_interes_ols = ['n_procedimientos', 'edad', 'severidad_grd', 'peso_relativo_grd']
    filas_ols_neo = [idx for idx in tabla_coef_neo.index
                    if any(v in str(idx) for v in vars_interes_ols)]
    display(Markdown('**Coeficientes OLS — Variables principales (Neoplasias):**'))
    display(tabla_coef_neo.loc[filas_ols_neo, ['coef','IC_inf','IC_sup','p_valor','sig']].round(4))
    tabla_coef_neo.to_csv(OUT_DIR / 'h3_neo_ols_coefs.csv')
    print('Tabla guardada: advance_3/outputs/h3_neo_ols_coefs.csv')


#### Interpretación — OLS Neoplasias

El coeficiente de `n_procedimientos` representa el impacto **independiente** de cada
procedimiento adicional sobre la duración de la estadía, **controlando** por edad,
severidad GRD, peso relativo y hospital.

Como la variable dependiente es `log(1 + dias_estada)`, los coeficientes se interpretan
como **semi-elasticidades**: β × 100 indica el cambio porcentual esperado en los días
de estadía por cada unidad adicional del predictor.

Por ejemplo, si β(n_procedimientos) = 0.08, se interpreta como:
*"cada procedimiento adicional se asocia con un aumento del 8% en los días de estadía
de pacientes oncológicos, manteniendo constantes la severidad y el hospital."*

> **TODO — Sebastián:**  
> Completa la interpretación del coeficiente real obtenido para Neoplasias.
> ¿El signo es el esperado (positivo = más procedimientos, más días)?
> ¿Cómo se compara el R² ajustado con el Pseudo-R² del modelo logístico?
> *Sección "Resultados — H3" del Word. Citar: White (1980) para errores robustos HC3.*


### 5.2 Sepsis — Regresión OLS


In [ ]:
print('=' * 65)
print('REGRESIÓN OLS — GRUPO SEPSIS')
print('=' * 65)
modelo_ols_sep, tabla_coef_sep = ajustar_ols(df_sep, VD_SEP, 'Sepsis')

if modelo_ols_sep is not None:
    filas_ols_sep = [idx for idx in tabla_coef_sep.index
                    if any(v in str(idx)
                           for v in ['n_procedimientos','edad','severidad_grd','peso_relativo_grd'])]
    display(Markdown('**Coeficientes OLS — Variables principales (Sepsis):**'))
    display(tabla_coef_sep.loc[filas_ols_sep, ['coef','IC_inf','IC_sup','p_valor','sig']].round(4))
    tabla_coef_sep.to_csv(OUT_DIR / 'h3_sep_ols_coefs.csv')
    print('Tabla guardada: advance_3/outputs/h3_sep_ols_coefs.csv')


In [ ]:
# ── Gráfico comparativo: coeficiente de n_procedimientos en OLS ───────────
grupos_ols = []
for nombre, tabla, modelo in [
    ('Neoplasia', tabla_coef_neo, modelo_ols_neo),
    ('Sepsis',    tabla_coef_sep, modelo_ols_sep),
]:
    if modelo is not None and 'n_procedimientos' in tabla.index:
        r = tabla.loc['n_procedimientos']
        grupos_ols.append({
            'grupo': nombre,
            'coef': r['coef'], 'IC_inf': r['IC_inf'], 'IC_sup': r['IC_sup'],
            'p': r['p_valor'], 'R2_adj': r['R2_ajustado']
        })

if grupos_ols:
    df_coef_comp = pd.DataFrame(grupos_ols)
    fig, ax = plt.subplots(figsize=(9, 4))
    colores = ['steelblue','coral']
    for i, row in df_coef_comp.iterrows():
        ax.errorbar(
            x=row['coef'], y=i,
            xerr=[[row['coef'] - row['IC_inf']], [row['IC_sup'] - row['coef']]],
            fmt='D', color=colores[i], markersize=10, capsize=5, linewidth=2
        )
        ax.text(row['IC_sup'] + 0.002, i,
                f"β={row['coef']:.4f} R²adj={row['R2_adj']:.3f} p={row['p']:.3e}",
                va='center', fontsize=9)
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
    ax.set_yticks(range(len(df_coef_comp)))
    ax.set_yticklabels(df_coef_comp['grupo'])
    ax.set_xlabel('Coeficiente OLS de n_procedimientos sobre log(1+días)')
    ax.set_title(
        'Impacto independiente de n_procedimientos sobre días de estadía\n'
        '(OLS con errores robustos HC3, efectos fijos por hospital, IC 95%)'
    )
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'h3_coef_ols_comparativo.png', dpi=150, bbox_inches='tight')
    plt.show()


> **TODO — Vicente:**  
> Compara el coeficiente β de n_procedimientos entre Neoplasias y Sepsis.
> ¿En cuál condición tiene mayor impacto sobre la duración de la hospitalización?
> ¿La diferencia en magnitud de los coeficientes sugiere perfiles clínicos distintos?
> Discute las implicancias para la gestión de camas en hospitales públicos chilenos.
> *APA 7: β = X.XX, IC95% [X.XX, X.XX], t(df) = X.XX, p < .001.*


---
## 6. Conclusiones Clínicas y Estadísticas


In [ ]:
# ── Tabla resumen de los tres modelos ─────────────────────────────────────
print('=' * 65)
print('RESUMEN EJECUTIVO — TRES HIPÓTESIS')
print('=' * 65)

for grupo, kw, logit_m, ols_m, t_or, t_coef in [
    ('Neoplasia', res_kw_neo, modelo_logit_neo, modelo_ols_neo, tabla_or_neo, tabla_coef_neo),
    ('Sepsis',    res_kw_sep, modelo_logit_sep, modelo_ols_sep, tabla_or_sep, tabla_coef_sep),
]:
    print(f'\n── {grupo} ──────────────────────────────────────────')

    # H1: Kruskal-Wallis
    h1_sig = 'SÍ' if kw['p'] < ALPHA else 'NO'
    print(f'  H1 (KW): H({kw["k"]}) = {kw["H"]:.2f}, p = {kw["p"]:.3e}, '
          f'ε² = {kw["epsilon2"]:.4f} → {h1_sig} significativo')

    # H2: Logit
    if logit_m is not None:
        psR2 = logit_m.prsquared
        if 'n_procedimientos' in t_or.index:
            r = t_or.loc['n_procedimientos']
            print(f'  H2 (Logit): OR(proc) = {r["OR"]:.4f}, IC95% [{r["IC_inf"]:.4f},{r["IC_sup"]:.4f}], '
                  f'p = {r["p_valor"]:.3e}, Pseudo-R² = {psR2:.4f}')
    else:
        print('  H2 (Logit): modelo no estimable')

    # H3: OLS
    if ols_m is not None:
        r2 = ols_m.rsquared_adj
        if 'n_procedimientos' in t_coef.index:
            r = t_coef.loc['n_procedimientos']
            print(f'  H3 (OLS):   β(proc) = {r["coef"]:.4f}, IC95% [{r["IC_inf"]:.4f},{r["IC_sup"]:.4f}], '
                  f'p = {r["p_valor"]:.3e}, R²adj = {r2:.4f}')
    else:
        print('  H3 (OLS): modelo no estimable')

print('\n' + '=' * 65)


### 6.1 Síntesis estadística

El análisis inferencial presentado en este avance responde a tres hipótesis centrales
sobre la variabilidad hospitalaria en el sistema público chileno, utilizando datos GRD
de egresos 2019–2024, separando los grupos diagnósticos de **Neoplasias** y **Sepsis**.

**Hipótesis 1 (Kruskal-Wallis + Dunn):** La prueba de Kruskal-Wallis detectó variabilidad
estadísticamente significativa en la intensidad de procedimientos (`n_procedimientos`) entre
hospitales en ambos grupos diagnósticos. El tamaño del efecto ε², calculado siguiendo la
propuesta de Tomczak & Tomczak (2014), cuantifica la magnitud de esta heterogeneidad.
El análisis post-hoc de Dunn con corrección de Bonferroni identificó los pares de hospitales
específicos que presentan diferencias significativas, permitiendo un diagnóstico granular
de dónde se concentra la variabilidad en la gestión clínica del sistema.

**Hipótesis 2 (Regresión Logística):** El modelo de regresión logística con efectos fijos
por hospital estima el impacto independiente del número de procedimientos sobre la probabilidad
de mortalidad intrahospitalaria, controlando por edad, severidad GRD y peso relativo.
Los Odds Ratios obtenidos para `n_procedimientos` divergen entre Neoplasias y Sepsis,
lo que sugiere que el perfil clínico del diagnóstico de base modifica sustantivamente
el rol de la intensidad terapéutica en el desenlace mortal.

**Hipótesis 3 (Regresión OLS con errores robustos HC3):** El modelo de regresión lineal
sobre `log(1 + dias_estada)` estima la semi-elasticidad de la estadía hospitalaria
respecto al número de procedimientos, después de controlar por severidad clínica
y efectos fijos de establecimiento. El R² ajustado indica la proporción de varianza
explicada por el modelo. La aplicación de errores robustos tipo HC3 (White, 1980)
garantiza inferencias válidas bajo heterocedasticidad.

---

> **TODO — Vicente:** Redacta el párrafo de implicancias para política pública de salud oncológica.
> ¿Los hospitales con mayor variabilidad en procedimientos corresponden a regiones con
> menor acceso a especialistas o menor presupuesto per cápita GRD?
> ¿Cómo podrían usarse estos resultados para diseñar protocolos estandarizados en el MINSAL?

> **TODO — José Tomás:** Redacta el párrafo de limitaciones metodológicas.
> Discute: (1) separación perfecta en el logit si algún hospital no tiene muertes;
> (2) endogeneidad potencial entre procedimientos y mortalidad;
> (3) sesgo de selección si los hospitales de mayor complejidad atienden casos más graves.

> **TODO — Sebastián:** Redacta el párrafo de próximas etapas.
> Propone: (1) modelos de efectos mixtos (lme4 / mixedlm) para capturar la estructura
> anidada paciente-hospital; (2) análisis de sensibilidad con diferentes cortes de GRD;
> (3) comparación pre/post pandemia 2019–2021 vs. 2022–2024.


In [ ]:
# ── Índice de archivos generados ─────────────────────────────────────────
print('ARCHIVOS GENERADOS EN advance_3/outputs/')
print('=' * 55)
if OUT_DIR.exists():
    for f in sorted(OUT_DIR.glob('*')):
        print(f'  {f.name:<45}  ({f.stat().st_size/1024:.1f} KB)')
else:
    print('  (directorio vacío — ejecutar notebook completo)')
